In [1]:
import os, glob, math, random, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from sklearn.model_selection import KFold
from tqdm.auto import tqdm

DATA_ROOT = "/kaggle/input/competitions/2026task3/detection"
TRAIN_IMG_DIR = os.path.join(DATA_ROOT, "train")
XML_DIR = os.path.join(DATA_ROOT, "train_location")
TEST_IMG_DIR = os.path.join(DATA_ROOT, "test")
GT_CSV = os.path.join(DATA_ROOT, "fovea_localization_train_GT.csv")
SAMPLE_SUB = os.path.join(DATA_ROOT, "sample_submission.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [2]:
def load_gt():
    df = pd.read_csv(GT_CSV)
    # data 列可能是 1,2,3... 对应 0001.jpg
    df["stem"] = df["data"].apply(lambda x: f"{int(x):04d}")
    return df.set_index("stem")

def load_bbox(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find("size")
    w = int(size.find("width").text); h = int(size.find("height").text)
    obj = root.find("object")
    if obj is None:
        return None, (w, h)
    bb = obj.find("bndbox")
    box = [float(bb.find("xmin").text), float(bb.find("ymin").text),
           float(bb.find("xmax").text), float(bb.find("ymax").text)]
    return box, (w, h)

gt = load_gt()
records = []
for stem in gt.index:
    xml_path = os.path.join(XML_DIR, f"{stem}.xml")
    img_path = os.path.join(TRAIN_IMG_DIR, f"{stem}.jpg")
    box, (w, h) = load_bbox(xml_path)
    fx, fy = gt.loc[stem, "Fovea_X"], gt.loc[stem, "Fovea_Y"]
    records.append(dict(stem=stem, img=img_path, box=box, W=w, H=h, fx=fx, fy=fy))
train_df = pd.DataFrame(records)
print(train_df.head())
print("样本数:", len(train_df))

   stem                                                img  \
0  0001  /kaggle/input/competitions/2026task3/detection...   
1  0002  /kaggle/input/competitions/2026task3/detection...   
2  0003  /kaggle/input/competitions/2026task3/detection...   
3  0004  /kaggle/input/competitions/2026task3/detection...   
4  0005  /kaggle/input/competitions/2026task3/detection...   

                                box     W     H           fx           fy  
0     [631.0, 903.0, 839.0, 1121.0]  1956  1934   734.031331  1016.817066  
1     [653.0, 901.0, 883.0, 1131.0]  1956  1934   769.024859  1018.059762  
2   [1368.0, 939.0, 1570.0, 1137.0]  2992  2000  1470.287475  1037.601074  
3  [1560.0, 1006.0, 1757.0, 1202.0]  2992  2000  1658.316671  1104.020062  
4  [1637.0, 1016.0, 1826.0, 1202.0]  2992  2000  1729.828431  1107.209978  
样本数: 80


In [3]:
train_df["cx"] = train_df["box"].apply(lambda b: (b[0]+b[2])/2)
train_df["cy"] = train_df["box"].apply(lambda b: (b[1]+b[3])/2)
train_df["dX"] = train_df["cx"] - train_df["fx"]
train_df["dY"] = train_df["cy"] - train_df["fy"]

print("ΔX  mean/std/max|.|:", train_df.dX.mean(), train_df.dX.std(), train_df.dX.abs().max())
print("ΔY  mean/std/max|.|:", train_df.dY.mean(), train_df.dY.std(), train_df.dY.abs().max())

# 全可见假设下，直接用 bbox 中心当坐标的 RMSE：
errs = np.concatenate([train_df.dX.values, train_df.dY.values])
rmse_center = np.sqrt(np.mean(errs**2))
print("bbox中心直接当坐标的训练RMSE:", rmse_center)

ΔX  mean/std/max|.|: 0.6068742500000113 1.3175620458223454 4.144571000000042
ΔY  mean/std/max|.|: 1.3160458750000033 1.8635619992075025 5.007329999999911
bbox中心直接当坐标的训练RMSE: 1.9031547173746677


In [4]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMG_SIZE = 800

def get_train_transform():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Rotate(limit=15, p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.ColorJitter(p=0.3),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.5))

def get_val_transform():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])

class FoveaDetDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.tf = get_train_transform() if train else get_val_transform()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = np.array(Image.open(r.img).convert("RGB"))
        if self.train:
            out = self.tf(image=img, bboxes=[r.box], labels=[1])
            boxes = torch.tensor(out['bboxes'], dtype=torch.float32)
        else:
            out = self.tf(image=img)
            sx, sy = IMG_SIZE / r.W, IMG_SIZE / r.H
            boxes = torch.tensor([[r.box[0]*sx, r.box[1]*sy, r.box[2]*sx, r.box[3]*sy]], dtype=torch.float32)
        target = {"boxes": boxes, "labels": torch.tensor([1], dtype=torch.int64)}
        return out['image'], target, torch.tensor([r.W, r.H], dtype=torch.float32)

In [5]:
def build_model():
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn_v2(weights=weights, box_score_thresh=0.05)
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, 2)

    # 冻结 backbone 前两个 stage，减少小数据过拟合
    for name, param in model.backbone.body.named_parameters():
        if 'layer1' in name or 'layer2' in name:
            param.requires_grad = False
    return model

In [6]:
def collate_fn(batch):
    imgs, tgts, sizes = zip(*batch)
    return list(imgs), list(tgts), list(sizes)

def train_one_epoch(model, loader, opt, epoch=None, fold=None):
    model.train()
    total = 0
    pbar = tqdm(loader, desc=f"fold{fold} ep{epoch}", leave=False)
    for imgs, tgts, _ in pbar:
        imgs = [im.to(DEVICE) for im in imgs]
        tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
        loss_dict = model(imgs, tgts)
        loss = sum(loss_dict.values())
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        total += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.3f}")
    return total / len(loader)

@torch.no_grad()
def eval_center_rmse(model, df):
    """全可见假设下，bbox中心当坐标的 RMSE（原图尺度），使用无增强 transform"""
    model.eval()
    errs = []
    for _, r in df.iterrows():
        img = np.array(Image.open(r.img).convert("RGB"))
        t = get_val_transform()(image=img)['image'].unsqueeze(0).to(DEVICE)
        out = model(t)[0]
        if len(out["boxes"]) == 0:
            errs += [r.W, r.H]
            continue
        b = out["boxes"][out["scores"].argmax()].cpu().numpy()
        cx = (b[0]+b[2])/2 / IMG_SIZE * r.W
        cy = (b[1]+b[3])/2 / IMG_SIZE * r.H
        errs += [cx - r.fx, cy - r.fy]
    return float(np.sqrt(np.mean(np.square(errs))))

def run_kfold(n_splits=5, epochs=25, bs=8, lr=1e-4):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_rmse = []
    fold_models = []
    for fold, (tr, va) in enumerate(kf.split(train_df)):
        tr_df, va_df = train_df.iloc[tr], train_df.iloc[va]
        ds_tr = FoveaDetDataset(tr_df, train=True)
        dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True,
                           collate_fn=collate_fn, num_workers=0)
        model = build_model().to(DEVICE)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

        best_rmse = float('inf')
        best_state = None

        for ep in range(epochs):
            tl = train_one_epoch(model, dl_tr, opt, epoch=ep+1, fold=fold)
            scheduler.step()
            r = eval_center_rmse(model, va_df)
            if r < best_rmse:
                best_rmse = r
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                print(f"fold{fold} ep{ep+1} loss={tl:.3f} val_rmse={r:.2f}  <== NEW BEST", flush=True)
            else:
                print(f"fold{fold} ep{ep+1} loss={tl:.3f} val_rmse={r:.2f}", flush=True)

        model.load_state_dict(best_state)
        model.to(DEVICE)
        torch.save(best_state, f"/kaggle/working/fold{fold}.pth")
        oof_rmse.append(best_rmse)
        fold_models.append(model)
        print(f"== fold{fold} BEST val_rmse={best_rmse:.2f} saved ==", flush=True)

    print("OOF RMSE:", np.mean(oof_rmse), oof_rmse)
    return fold_models

fold_models = run_kfold()

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


  0%|          | 0.00/167M [00:00<?, ?B/s]

 11%|█         | 17.9M/167M [00:00<00:00, 187MB/s]

 26%|██▌       | 42.9M/167M [00:00<00:00, 231MB/s]

 41%|████      | 67.8M/167M [00:00<00:00, 244MB/s]

 55%|█████▌    | 92.6M/167M [00:00<00:00, 251MB/s]

... (256 outputs left out)

fold4 ep25:   0%|          | 0/8 [00:00<?, ?it/s]

fold4 ep25 loss=0.027 val_rmse=906.45


== fold4 BEST val_rmse=16.90 saved ==


OOF RMSE: 13.960286903381348 [14.762157440185547, 15.888968467712402, 4.581577301025391, 17.664438247680664, 16.904293060302734]


In [7]:
@torch.no_grad()
def predict_one(model, img_path):
    img = np.array(Image.open(img_path).convert("RGB"))
    h, w = img.shape[:2]
    t = get_val_transform()(image=img)['image'].unsqueeze(0).to(DEVICE)
    out = model(t)[0]
    if len(out["boxes"]) == 0:
        return None, 0.0
    idx = out["scores"].argmax()
    b = out["boxes"][idx].cpu().numpy()
    score = float(out["scores"][idx].cpu())
    cx = (b[0]+b[2])/2 / IMG_SIZE * w
    cy = (b[1]+b[3])/2 / IMG_SIZE * h
    return (cx, cy), score

def ensemble_predict(models, img_path):
    preds, scores = [], []
    for m in models:
        m.eval()
        p, s = predict_one(m, img_path)
        if p is not None:
            preds.append(p); scores.append(s)
    if not preds:
        return (0.0, 0.0), 0.0
    return tuple(np.mean(preds, axis=0)), float(np.mean(scores))

def make_submission(models, tau=0.5, out="submission.csv"):
    test_imgs = sorted(glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg")))
    rows = []
    for p in test_imgs:
        stem = os.path.splitext(os.path.basename(p))[0]   # 0081
        num = int(stem)                                    # 81
        (cx, cy), score = ensemble_predict(models, p)
        if score < tau:
            cx, cy = 0.0, 0.0
        rows.append((f"{num}_Fovea_X", cx))
        rows.append((f"{num}_Fovea_Y", cy))
    sub = pd.DataFrame(rows, columns=["ImageID", "value"])
    # 严格按 sample_submission 顺序对齐
    ss = pd.read_csv(SAMPLE_SUB)
    ss = ss.set_index("ImageID").join(sub.set_index("ImageID"))
    ss["value"] = ss["value"].fillna(0.0)
    ss.reset_index().to_csv(out, index=False)
    print("saved", out)
    return sub

# 先看测试集 score 分布，辅助定 tau
def score_distribution(models):
    test_imgs = sorted(glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg")))
    ss = []
    for p in tqdm(test_imgs, desc="scoring test"):
        _, s = ensemble_predict(models, p)
        ss.append((os.path.basename(p), s))
    df = pd.DataFrame(ss, columns=["file","score"]).sort_values("score")
    print(df.to_string(index=False))
    return df

sd = score_distribution(fold_models)

scoring test:   0%|          | 0/20 [00:00<?, ?it/s]

    file    score
0086.jpg 0.416202
0081.jpg 0.780516
0091.jpg 0.929265
0096.jpg 0.943649
0084.jpg 0.962333
0092.jpg 0.972496
0093.jpg 0.976109
0097.jpg 0.976795
0088.jpg 0.977445
0094.jpg 0.977689
0085.jpg 0.977859
0099.jpg 0.978179
0089.jpg 0.978202
0087.jpg 0.978694
0082.jpg 0.978971
0090.jpg 0.979001
0095.jpg 0.979207
0100.jpg 0.979279
0098.jpg 0.981553
0083.jpg 0.981902


In [8]:


@torch.no_grad()
def ensemble_predict_fast(models, img_path):
    img = np.array(Image.open(img_path).convert("RGB"))
    h, w = img.shape[:2]
    t = get_val_transform()(image=img)['image'].unsqueeze(0).to(DEVICE)
    preds, scores = [], []
    for m in models:
        m.eval()
        out = m(t)[0]
        if len(out["boxes"]) == 0:
            continue
        idx = out["scores"].argmax()
        b = out["boxes"][idx].cpu().numpy()
        preds.append(((b[0]+b[2])/2 / IMG_SIZE * w, (b[1]+b[3])/2 / IMG_SIZE * h))
        scores.append(float(out["scores"][idx].cpu()))
    if not preds:
        return (0.0, 0.0), 0.0
    return tuple(np.mean(preds, axis=0)), float(np.mean(scores))

def make_submission(models, tau=0.0, out="submission.csv"):
    test_imgs = sorted(glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg")))
    assert len(test_imgs) > 0, "测试图没找到，检查 TEST_IMG_DIR"
    rows = []
    for p in test_imgs:
        stem = os.path.splitext(os.path.basename(p))[0]
        num = int(stem)
        (cx, cy), score = ensemble_predict_fast(models, p)
        if score < tau:
            cx, cy = 0.0, 0.0
        rows.append((f"{num}_Fovea_X", cx))
        rows.append((f"{num}_Fovea_Y", cy))
        print(f"{stem}: score={score:.3f} pred=({cx:.1f},{cy:.1f})", flush=True)

    sub = pd.DataFrame(rows, columns=["ImageID", "value"])
    # 用 sample 的 ImageID 顺序对齐，避免 value 列冲突
    ss = pd.read_csv(SAMPLE_SUB)[["ImageID"]]
    ss = ss.merge(sub, on="ImageID", how="left")
    ss["value"] = ss["value"].fillna(0.0)
    ss.to_csv(out, index=False)

    assert os.path.exists(out), "文件未生成"
    check = pd.read_csv(out)
    print("=== 生成成功 ===")
    print("路径:", os.path.abspath(out))
    print("行数:", len(check), "(应为 40)")
    print("空值:", check["value"].isna().sum(), "(应为 0)")
    print(check.head(6))
    return check

# 先确保 fold_models 存在
print("模型数:", len(fold_models))

# 生成基线提交（全可见）
check = make_submission(fold_models, tau=0.0, out="submission.csv")

模型数: 5


0081: score=0.781 pred=(1739.7,1161.2)


0082: score=0.979 pred=(1685.0,1103.4)


0083: score=0.982 pred=(1487.7,1040.5)


0084: score=0.962 pred=(1237.0,1112.7)


0085: score=0.978 pred=(1199.2,980.0)


... (12 outputs left out)

0098: score=0.982 pred=(1471.4,1068.6)


0099: score=0.978 pred=(1450.9,1033.6)


0100: score=0.979 pred=(1679.1,1091.2)


=== 生成成功 ===
路径: /kaggle/working/submission.csv
行数: 40 (应为 40)
空值: 0 (应为 0)
      ImageID      value
0  81_Fovea_X  1739.7096
1  81_Fovea_Y  1161.2283
2  82_Fovea_X  1685.0016
3  82_Fovea_Y  1103.4358
4  83_Fovea_X  1487.6873
5  83_Fovea_Y  1040.5166


In [9]:
class OffsetNet(nn.Module):
    def __init__(self):
        super().__init__()
        from torchvision.models import resnet18
        self.backbone = resnet18(weights="DEFAULT")
        self.backbone.fc = nn.Linear(512, 2)  # 预测 bbox 中心到真值的偏移(归一化)
    def forward(self, x): return self.backbone(x)

def crop_bbox(img, box, pad=0.2):
    W, H = img.size
    x1,y1,x2,y2 = box
    w, h = x2-x1, y2-y1
    x1 -= pad*w; y1 -= pad*h; x2 += pad*w; y2 += pad*h
    x1,y1 = max(0,x1), max(0,y1); x2,y2 = min(W,x2), min(H,y2)
    return img.crop((x1,y1,x2,y2)), (x1,y1,x2,y2)

In [10]:
# 跑完训练后，加这段
@torch.no_grad()
def find_hard_samples(models, df):
    """找每折验证集里误差最大的样本"""
    results = []
    for fold, (tr, va) in enumerate(KFold(5, shuffle=True, random_state=SEED).split(train_df)):
        va_df = train_df.iloc[va]
        m = models[fold]
        for _, r in va_df.iterrows():
            img = np.array(Image.open(r.img).convert("RGB"))
            t = get_val_transform()(image=img)['image'].unsqueeze(0).to(DEVICE)
            out = m(t)[0]
            if len(out["boxes"]) == 0:
                err = 9999
            else:
                b = out["boxes"][out["scores"].argmax()].cpu().numpy()
                cx = (b[0]+b[2])/2 / IMG_SIZE * r.W
                cy = (b[1]+b[3])/2 / IMG_SIZE * r.H
                err = np.sqrt((cx - r.fx)**2 + (cy - r.fy)**2)
            results.append((r.stem, err, r.img))
    results.sort(key=lambda x: -x[1])
    print("误差最大的 5 张训练图：")
    for stem, err, path in results[:5]:
        print(f"  {stem}: {err:.1f} px  {path}")
    return results

hard = find_hard_samples(fold_models, train_df)

误差最大的 5 张训练图：
  0049: 69.4 px  /kaggle/input/competitions/2026task3/detection/train/0049.jpg
  0034: 64.9 px  /kaggle/input/competitions/2026task3/detection/train/0034.jpg
  0072: 62.2 px  /kaggle/input/competitions/2026task3/detection/train/0072.jpg
  0062: 58.6 px  /kaggle/input/competitions/2026task3/detection/train/0062.jpg
  0016: 51.4 px  /kaggle/input/competitions/2026task3/detection/train/0016.jpg


In [11]:
import torch
print("GPU 数量:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), 
          f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB")

GPU 数量: 2
0 Tesla T4 15.6 GB
1 Tesla T4 15.6 GB


In [12]:
import os, gc, json, time
import numpy as np, pandas as pd, torch
from PIL import Image
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold

# 这个函数在子进程里跑，必须把用到的都放进来或全局可见
def train_one_fold(fold, tr_idx, va_idx, config, gpu_id, save_dir):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    import torch
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # 用全局的 train_df / 类 / 函数（notebook 里 fork 模式下可访问）
    tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
    va_df = train_df.iloc[va_idx].reset_index(drop=True)

    ds_tr = FoveaDetDataset(tr_df, train=config["use_aug"])
    dl_tr = DataLoader(ds_tr, batch_size=config["bs"], shuffle=True,
                       collate_fn=collate_fn, num_workers=0)
    model = build_model(freeze=config["freeze"]).to(device)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=config["lr"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=config["epochs"])

    best, best_state, history = float("inf"), None, []
    for ep in range(config["epochs"]):
        model.train()
        total = 0
        for imgs, tgts, _ in dl_tr:
            imgs = [im.to(device) for im in imgs]
            tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]
            loss = sum(model(imgs, tgts).values())
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); total += loss.item()
        sched.step()

        # 评估（用无增强 transform）
        model.eval()
        errs = []
        with torch.no_grad():
            for _, r in va_df.iterrows():
                img = np.array(Image.open(r.img).convert("RGB"))
                t = get_val_transform()(image=img)['image'].unsqueeze(0).to(device)
                out = model(t)[0]
                if len(out["boxes"]) == 0:
                    errs += [r.W, r.H]; continue
                b = out["boxes"][out["scores"].argmax()].cpu().numpy()
                cx = (b[0]+b[2])/2 / IMG_SIZE * r.W
                cy = (b[1]+b[3])/2 / IMG_SIZE * r.H
                errs += [cx - r.fx, cy - r.fy]
        rmse = float(np.sqrt(np.mean(np.square(errs))))
        history.append(rmse)
        if rmse < best:
            best = rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    path = os.path.join(save_dir, f"{config['name']}_fold{fold}.pth")
    torch.save(best_state, path)
    del model; gc.collect(); torch.cuda.empty_cache()
    return {"config": config["name"], "fold": fold, "best_rmse": best,
            "history": history, "ckpt": path}

In [13]:
def build_model(freeze=True):
    from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT, box_score_thresh=0.05)
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, 2)
    if freeze:
        for name, p in model.backbone.body.named_parameters():
            if 'layer1' in name or 'layer2' in name:
                p.requires_grad = False
    return model

In [14]:
from concurrent.futures import ProcessPoolExecutor
import itertools

SAVE_DIR = "/kaggle/working/exp"
os.makedirs(SAVE_DIR, exist_ok=True)

# 4 组对照：每次只改一个变量
EXPERIMENTS = [
    {"name": "E1_noaug",      "use_aug": False, "freeze": False, "bs": 4, "epochs": 15, "lr": 1e-4},
    {"name": "E2_aug",        "use_aug": True,  "freeze": False, "bs": 4, "epochs": 15, "lr": 1e-4},
    {"name": "E3_aug_freeze", "use_aug": True,  "freeze": True,  "bs": 4, "epochs": 15, "lr": 1e-4},
    {"name": "E4_aug_freeze_bs8", "use_aug": True, "freeze": True, "bs": 8, "epochs": 15, "lr": 1e-4},
]

# 每折索引
kf = KFold(5, shuffle=True, random_state=42)
FOLDS = list(kf.split(train_df))

def run_all(experiments, n_gpu=2, n_workers=None):
    """把 (experiment, fold) 任务丢进进程池，自动分配 GPU"""
    tasks = []
    for exp in experiments:
        for fold, (tr, va) in enumerate(FOLDS):
            tasks.append((exp, fold, tr, va))
    n_workers = n_workers or n_gpu
    results = []
    # gpu 轮转
    def gpu_of(i): return i % n_gpu

    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futures = []
        for i, (exp, fold, tr, va) in enumerate(tasks):
            g = gpu_of(i)
            futures.append(ex.submit(train_one_fold, fold, tr, va, exp, g, SAVE_DIR))
        from tqdm import tqdm
        for f in tqdm(futures, desc="训练中"):
            results.append(f.result())
    return results

In [ ]:
import time
t0 = time.time()
results = run_all(EXPERIMENTS, n_gpu=2)
print(f"总耗时 {time.time()-t0:.1f}s")

# 汇总成表
df_res = pd.DataFrame([{"exp": r["config"], "fold": r["fold"], "best_rmse": r["best_rmse"]} for r in results])
pivot = df_res.pivot_table(index="exp", columns="fold", values="best_rmse")
pivot["OOF"] = df_res.groupby("exp")["best_rmse"].mean()
print(pivot.round(2))

# 保存
df_res.to_csv("/kaggle/working/exp_results.csv", index=False)
pivot.to_csv("/kaggle/working/exp_pivot.csv")

In [ ]:
# 选出 OOF 最低的配置
best_exp = pivot["OOF"].idxmin()
print("最优配置:", best_exp, "OOF:", pivot.loc[best_exp, "OOF"])

# 加载该配置的 5 个模型
cfg = [e for e in EXPERIMENTS if e["name"] == best_exp][0]
models = []
for fold in range(5):
    m = build_model(freeze=cfg["freeze"]).to(DEVICE)
    m.load_state_dict(torch.load(f"/kaggle/working/exp/{best_exp}_fold{fold}.pth", map_location=DEVICE))
    m.eval(); models.append(m)

# 复用之前的 ensemble + make_submission
check = make_submission(models, tau=0.0, out="submission_best.csv")